# Thuc nghiem toi thieu khoa luan — 9 run con lai

Doi `JOB` o Cell 2 roi Run All. **Moi lan chay DUNG mot job.**

| JOB | Noi dung | Config | ~gio |
|---|---|---|---|
| `fmi_m6`   | Forget-MI, MIMIC 6%  | baseline | 3,0 |
| `p3_m6`    | P3-NoKD-More, MIMIC 6%  | advanced | 2,7 |
| `fmi_m10`  | Forget-MI, MIMIC 10% | baseline | 3,0 |
| `p3_m10`   | P3-NoKD-More, MIMIC 10% | advanced | 2,7 |
| `fmi_iu`   | Forget-MI, IU 3%     | baseline_iu | 2,5 |
| `p3_iu`    | P3-NoKD-More, IU 3%  | loku_iu  | 2,2 |
| `abl_fila` | P3 w/o Fisher/FILA, MIMIC 3% | advanced | 2,6 |
| `abl_ihl`  | P3 w/o IHL, MIMIC 3%         | advanced | 2,7 |
| `abl_mumr` | P3 w/o MU/MR, MIMIC 3%       | advanced | 2,7 |

**MIMIC 3% (fmi + p3more) DA CHAY XONG — khong chay lai.**

Uoc tinh gio la SUY RA tu run 3% (P3 9606s = 2,67h; FMI 6260s + 1046s chan doan = 2,03h;
FMI o day cong them CE-selector nen ~3h). Khong phai so do that cho 6/10%/IU.

## Chi so bao cao (theo danh sach thuc nghiem)
Chi dung **S2 (Closest CE)** + **E30**. S1/S3/S4 van duoc ghi ra file nhung KHONG bao cao.
Bang chinh chi lay: Df-AUC/F1, Dt-AUC/F1, MIA, Forget-CE, Test-CE, tham so, **T_core**, GPU peak.
KHONG dua T_selection / T_eval / pipeline vao bang chinh.


In [ ]:
# Cell 1: setup + CHOT CHAN code da push
import os, subprocess
WORK='/kaggle/working'; REPO=f'{WORK}/Forget-MI-LoKU'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/nhnhu146/Forget-MI-LoKU.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
os.chdir(REPO)
assert os.path.exists('training/adv_common.py'),'push code truoc + re-import notebook'
_adv=open('training/adv_common.py').read()
assert 'ce_selector' in _adv and 'checkpoint_selection_' in _adv, \
    '❌ adv_common CHUA co hook CE-selector -> chay `git push` code MOI roi moi Save Version!'
assert 'OnlineCESelector' in open('training/ce_selector_pilot.py').read(), '❌ git push code moi truoc!'
assert os.path.exists('training/forgetmi_p3_cand.py'), '❌ chua push forgetmi_p3_cand.py!'
print('✅ Code CE-selector da co (hook + OnlineCESelector).')
subprocess.run(['pip','install','-q','pydicom','scikit-image','scikit-learn','pyyaml','wandb','seaborn==0.13.2'],check=True)
subprocess.run(['pip','install','-q','transformers==4.38.0','peft==0.10.0','accelerate==0.27.0'],check=True)
import torch; assert torch.cuda.is_available(),'Bat GPU'
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GPU   :',torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: CHON JOB + path discovery
import glob, os

# --- DOI DUNG 1 DONG NAY ---
JOB = 'fmi_m6'   # fmi_m6|p3_m6|fmi_m10|p3_m10|fmi_iu|p3_iu|abl_fila|abl_ihl|abl_mumr

SEED   = 42
EPOCHS = 30

# job -> (dataset, forget%, phuong phap, ablation)
JOBS = {
 'fmi_m6'  : ('mimic', 6,  'fmi', 'none'),
 'p3_m6'   : ('mimic', 6,  'p3',  'none'),
 'fmi_m10' : ('mimic', 10, 'fmi', 'none'),
 'p3_m10'  : ('mimic', 10, 'p3',  'none'),
 'fmi_iu'  : ('iu',    3,  'fmi', 'none'),
 'p3_iu'   : ('iu',    3,  'p3',  'none'),
 'abl_fila': ('mimic', 3,  'p3',  'fisher_fila'),
 'abl_ihl' : ('mimic', 3,  'p3',  'ihl'),
 'abl_mumr': ('mimic', 3,  'p3',  'mu_mr'),
}
assert JOB in JOBS, f'JOB phai thuoc {sorted(JOBS)}'
DATASET, FORGET_PCT, KIND, ABLATE = JOBS[JOB]

def fd(*slugs):
    for s in slugs:
        if os.path.isdir(f'/kaggle/input/{s}'): return f'/kaggle/input/{s}'
        h=glob.glob(f'/kaggle/input/datasets/*/{s}')
        if h: return sorted(h)[0]
    return None
def bins(root): return sorted(glob.glob(os.path.join(root,'**','pytorch_model.bin'),recursive=True),key=len)
def first_existing(root, rels):
    for r in rels:
        p=os.path.join(root,r)
        if os.path.exists(p): return p
    return None

tag=f'{DATASET}{FORGET_PCT}per'
OUT=f'/kaggle/working/kltn_{JOB}_s{SEED}'
RESULTS=f'/kaggle/working/results_{JOB}.csv'

if DATASET=='mimic':
    CONFIG = 'config_baseline_kaggle.yaml' if KIND=='fmi' else 'config_advanced_kaggle.yaml'
    DATA=fd('forget-mi-data'); MOD=fd('forget-mi-models-full','forget-mi-models')
    assert DATA and MOD,'Add forget-mi-data + forget-mi-models-full'
    BASE=os.path.dirname([b for b in bins(MOD) if 'training_original_model' in b][0])
    gh=[b for b in bins(MOD) if f'model_retrained_{FORGET_PCT}per' in b]
    GOLD=os.path.dirname(gh[0]) if gh else BASE; HAS_GOLD=bool(gh)
    TEXT=os.path.join(DATA,'data','metadata'); IMG=os.path.join(DATA,'data','img_data')
    SPLIT='./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv'
    FORGET=f'./data_splits/forget_set_{FORGET_PCT}per.csv'
else:
    CONFIG = 'config_baseline_iu_kaggle.yaml' if KIND=='fmi' else 'config_loku_iu_kaggle.yaml'
    DATA=fd('forget-mi-data-iu'); MOD=fd('forget-mi-models-iu'); MODRE=fd('forget-mi-models-iu-re')
    RAD=fd('chest-xrays-indiana-university')
    assert DATA and MOD and RAD,'Add forget-mi-data-iu + forget-mi-models-iu + forget-mi-models-iu-re + chest-xrays-indiana-university'
    ogb=[b for b in bins(MOD) if 'model_og' in b.lower() or 'base_model' in b.lower()] or bins(MOD)
    BASE=os.path.dirname(ogb[0])
    reb=(bins(MODRE) if MODRE else []) or [b for b in bins(MOD) if 'retrain' in b.lower()]
    GOLD=os.path.dirname(reb[0]) if reb else BASE; HAS_GOLD=bool(reb)
    tsv=glob.glob(os.path.join(DATA,'**','all_data.tsv'),recursive=True) or glob.glob('/kaggle/input/**/all_data.tsv',recursive=True)
    TEXT=os.path.dirname(tsv[0]) if tsv else first_existing(DATA,['data/metadata','metadata'])
    IMG=first_existing(DATA,['data/img_data','img_data']) or (first_existing(RAD,['images/images_normalized','images']) if RAD else None) or RAD
    sp=glob.glob(os.path.join(DATA,'**','iu-split.csv'),recursive=True) or glob.glob('/kaggle/input/**/iu-split.csv',recursive=True) or glob.glob(os.path.join(DATA,'**','*iu*split*.csv'),recursive=True)
    fg=glob.glob(os.path.join(DATA,'**',f'forget_set_{FORGET_PCT}per_iu.csv'),recursive=True) or glob.glob(f'/kaggle/input/**/forget_set_{FORGET_PCT}per_iu.csv',recursive=True)
    assert sp and fg,'Khong thay iu-split / forget_set_iu'
    SPLIT=sp[0]; FORGET=fg[0]

for n,p in {'BASE':BASE,'TEXT':TEXT,'IMG':IMG,'SPLIT':SPLIT,'FORGET':FORGET}.items():
    assert p and os.path.exists(p),f'Missing {n}: {p}'

COMMON={'forget_set_path':FORGET,'base_model_path':BASE,'bert_pretrained_dir':BASE,
        'retrained_model_path':GOLD,'text_data_dir':TEXT,'img_data_dir':IMG,
        'data_split_path':SPLIT,'results_csv_path':RESULTS,'use_noise':1}
# P3-NoKD-More: cau hinh DA KHOA tu MIMIC 3%, khong tuning lai theo 6/10%/IU
MLP_TXT='attention.output.dense|intermediate.dense|output.dense'
MORE={'lora_extra_target_modules':MLP_TXT,'lora_image_last_k_blocks':2}

RID=f'{JOB}_s{SEED}'; OD=f'{OUT}/{RID}'
print('JOB',JOB,'| DATASET',DATASET,FORGET_PCT,'% | KIND',KIND,'| ABLATE',ABLATE)
print('config',CONFIG,'| GOLD',HAS_GOLD,'| run id',RID)
print('BASE',BASE); print('FORGET',FORGET)


In [ ]:
# Cell 3: CHAY (30 epoch). Ca hai deu bat CE-selector de co S2.
import os, subprocess, time
env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled',
     'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}

if KIND=='p3':
    ovr=dict(COMMON); ovr.update(MORE)
    ovr.update({'id':RID,'output_dir':OD,'unlearn_epochs':EPOCHS,
                'ce_selector':1,'s4_delta':0.15,
                'history_csv_path':f'/kaggle/working/perepoch_{RID}.csv'})
    cmd=['python','training/forgetmi_p3_cand.py','--config',CONFIG,'--seed',str(SEED),
         '--scheme','uni_nokd','--ablate',ABLATE,'--fresh','--override',
         ','.join(f'{k}={v}' for k,v in ovr.items())]
else:
    ovr=dict(COMMON)
    ovr.update({'id':RID,'output_dir':OD,'unlearn_epochs':EPOCHS,
                'evaluate_last_and_best':1,
                'ce_selector_out':f'{OD}/checkpoint_selection_forgetmi',
                'history_csv_path':f'/kaggle/working/perepoch_{RID}.csv'})
    cmd=['python','training/forgetmi_partial.py','--config',CONFIG,'--seed',str(SEED),
         '--fresh','--override',','.join(f'{k}={v}' for k,v in ovr.items())]

print('='*72+f'\n{RID}\n'+'='*72)
t0=time.time()
try:
    subprocess.run(cmd,env=env,check=True); print(f'OK {RID}  wall {(time.time()-t0)/3600:.2f}h')
except subprocess.CalledProcessError as e:
    print('FAIL',RID,'rc=',e.returncode)


In [ ]:
# Cell 4: eval OG + GOLD cho MUC QUEN NAY (chi can chay 1 lan moi muc quen)
# 3% da co roi -> de RUN_REF=False. 6%, 10%, IU: bat True o DUNG MOT job cua muc do.
import os, subprocess
RUN_REF = (FORGET_PCT!=3 or DATASET!='mimic')

P3CFG = 'config_advanced_kaggle.yaml' if DATASET=='mimic' else 'config_loku_iu_kaggle.yaml'
def evalref(label, mpath):
    ovr=dict(COMMON); ovr['output_dir']=f'{OUT}/_ref'; ovr['results_csv_path']=RESULTS
    env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled'}
    cmd=['python','training/forgetmi_eval_only.py','--config',P3CFG,'--seed',str(SEED),
         '--label',label,'--model_type','pretrained','--model_path',mpath,
         '--method','reference','--override',','.join(f'{k}={v}' for k,v in ovr.items())]
    print('eval-ref',label)
    try: subprocess.run(cmd,env=env,check=True)
    except subprocess.CalledProcessError as e: print('FAIL',label,e.returncode)

if RUN_REF:
    evalref(f'og_{tag}',BASE)
    if HAS_GOLD: evalref(f're_{tag}',GOLD)
    else: print('(khong co GOLD cho',tag,')')
else:
    print('RUN_REF=False -> OG/GOLD cua MIMIC 3% da co tu run truoc')


In [ ]:
# Cell 5: S2 + E30 + T_core + GPU peak — DUNG cac so nay cho bang Chuong 4
import glob, json, os, pandas as pd
pd.set_option('display.width',220)

# --- S2 (gold-free) ---
for f in sorted(glob.glob(f'{OUT}/**/selected_checkpoints.json',recursive=True)):
    d=json.load(open(f,encoding='utf-8'))['results']
    s2=d.get('S2_closest_ce',{})
    print('S2 (Closest CE) ->', os.path.basename(os.path.dirname(f)))
    if s2.get('epoch') is None:
        print('   khong chon duoc:',s2.get('note'))
    else:
        print(f"   E{s2['epoch']}  Df-AUC {s2['Df_AUC']}  Df-F1 {s2['Df_F1']}  "
              f"Dt-AUC {s2['Dt_AUC']}  Dt-F1 {s2['Dt_F1']}  MIA {s2['MIA']}  "
              f"fce {s2['forget_ce']}  nmval_ce {s2['nm_val_ce']}")

# --- E30 + tham chieu ---
if os.path.exists(RESULTS):
    print('\n===== E30 (last) + OG/GOLD =====')
    dr=pd.read_csv(RESULTS)
    cols=[c for c in ['id','method','checkpoint_kind','checkpoint','selected_epoch',
                      'Forget_AUC','Forget_Macro_F1','Test_AUC','Test_Macro_F1',
                      'Df_AUC','Df_F1','Dt_AUC','Dt_F1','MIA','forget_ce','test_ce',
                      'trainable_params','trainable_ratio'] if c in dr.columns]
    print(dr[cols].to_string(index=False))

# --- T_core + GPU peak ---
print('\n===== T_core + GPU peak =====')
for f in sorted(glob.glob(f'{OUT}/**/timing_*.json',recursive=True)):
    d=json.load(open(f,encoding='utf-8'))
    print(f"{d.get('method'):18} T_fisher {d.get('fisher_seconds',0):7.1f}s  "
          f"T_fila {d.get('fila_seconds',0):6.1f}s  T_train {d.get('train_seconds',0):8.1f}s  "
          f"=> T_core {d.get('core_seconds',0):8.1f}s")
    print(f"{'':18} peak {d.get('core_peak_allocated_gb',0):.2f} GB alloc / "
          f"{d.get('core_peak_reserved_gb',0):.2f} GB reserved  |  "
          f"params {d.get('trainable_params',0):,} ({100*d.get('trainable_ratio',0):.2f}%)")
    print(f"{'':18} GPU {d.get('gpu_name')}  selector {d.get('selector')}")

print('\nTAI VE: timing_*.json + selected_checkpoints.json + results_*.csv + perepoch_*.csv')
